# **Khám phá nhanh dữ liệu**

---

## **Mục tiêu**

Thực hiện **Data Profiling** — khám phá toàn bộ 15 file CSV gốc để hiểu data trông như thế nào trước khi làm bất cứ điều gì.

**Nguyên tắc xuyên suốt:**
- Không clean, không sửa, không assume gì cả trong notebook này.
- Không cross-check với đề bài hay kết quả MCQ.
- Chỉ in ra **data thực tế đang có gì** để nghiên cứu và quyết định bước clean tiếp theo.

**Cấu trúc notebook:**

| Cell | Nội dung |
|------|----------|
| 0 | Setup — thư viện, đường dẫn, report buffer |
| 1 | Load 15 file (thuần, không parse_dates) |
| 2 | Schema và dtype từng bảng |
| 3 | Missing values |
| 4 | Categorical columns — toàn bộ unique values |
| 5 | High-cardinality string columns — top/bottom 10 |
| 6 | Numeric columns — describe + n_zeros + n_negative |
| 7 | Date columns — range, parse failures, ngày trong tương lai |
| 8 | Primary key candidates |
| 9 | Foreign key orphans |
| 10 | Cardinality thực tế giữa các bảng |
| 11 | Full duplicate rows |
| 12 | Sanity checks theo logic thuần |
| 13 | Crosstab shipments / returns theo order_status |
| 14 | Gap detection trong time series sales.csv |
| 15 | Lưu report ra data/interim/profiling_report.md |

**Output:** `data/interim/profiling_report.md` — file markdown để cả nhóm đọc chung, highlight các bất thường, rồi mới viết `clean_data.py`.

---
## Cell 0 — Setup

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Chuyển RAW_DIR thành đường dẫn web trực tiếp đến thư mục raw trên nhánh develop
RAW_DIR = "https://raw.githubusercontent.com/phuonganh85/Datathon2026_DataFusion4/develop/data/raw/"

# OUT_DIR giữ nguyên là đường dẫn local để lưu file sau khi xử lý
OUT_DIR = Path('../data/interim')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Buffer để collect mọi output rồi ghi ra file markdown cuối notebook
REPORT_BUFFER = StringIO()

def log(*args, **kwargs):
    """In ra console đồng thời ghi vào buffer markdown."""
    print(*args, **kwargs)
    print(*args, **kwargs, file=REPORT_BUFFER)

log('# Data Profiling Report')
log(f'\nGenerated at: {pd.Timestamp.now()}\n')

# Data Profiling Report

Generated at: 2026-05-01 01:57:19.465233



---
## Cell 1 — Load 15 file CSV

Load **thuần** — không dùng `parse_dates`, không hint `dtype`. Để pandas tự suy luận kiểu dữ liệu, vì chính cách pandas suy luận sẽ tiết lộ data có vấn đề gì (ví dụ: cột date bị đọc thành `object` tức là có giá trị không hợp lệ bên trong).

In [3]:
FILES = [
    'products', 'customers', 'promotions', 'geography',
    'orders', 'order_items', 'payments', 'shipments',
    'returns', 'reviews', 'sales', 'inventory', 'web_traffic'
]

tables = {}
for name in FILES:
    # 1. Dùng f-string để nối chuỗi URL thay vì dùng toán tử chia (/)
    path = f"{RAW_DIR}{name}.csv"
    
    # 2. Dùng try...except để đọc file từ web, bỏ qua file lỗi
    try:
        tables[name] = pd.read_csv(path, low_memory=False)
    except Exception:
        log(f'  [MISSING] File không tồn tại hoặc lỗi tải: {path}')

log(f'\n## 1. Loaded {len(tables)} tables\n')
log('| Table | Rows | Cols | Memory (MB) |')
log('|-------|------|------|-------------|')
for name, df in tables.items():
    mem_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    log(f'| {name} | {len(df):,} | {df.shape[1]} | {mem_mb:.1f} |')


## 1. Loaded 13 tables

| Table | Rows | Cols | Memory (MB) |
|-------|------|------|-------------|
| products | 2,412 | 8 | 0.7 |
| customers | 121,930 | 7 | 35.0 |
| promotions | 50 | 10 | 0.0 |
| geography | 39,948 | 4 | 6.9 |
| orders | 646,945 | 8 | 194.2 |
| order_items | 714,669 | 7 | 78.0 |
| payments | 646,945 | 4 | 50.6 |
| shipments | 566,067 | 4 | 72.3 |
| returns | 39,939 | 7 | 8.0 |
| reviews | 113,551 | 7 | 23.3 |
| sales | 3,833 | 3 | 0.3 |
| inventory | 60,247 | 17 | 19.7 |
| web_traffic | 3,652 | 7 | 0.6 |


**Kết luận Cell 1:** Đọc bảng output để xác nhận đủ 13 bảng được load. Chú ý cột Memory để biết bảng nào nặng — ảnh hưởng đến thứ tự xử lý sau này.

---
## Cell 2 — Schema và dtype từng bảng

In `dtypes` của tất cả các cột. Thêm heuristic phát hiện cột tên có `date` nhưng pandas đọc thành `object` — đây là tín hiệu cột đó có giá trị không thể parse thành ngày.

In [4]:
log('\n## 2. Schema và Dtypes\n')

for name, df in tables.items():
    log(f'\n### `{name}.csv`')
    log(f'```\n{df.dtypes.to_string()}\n```')

    # Heuristic: tên cột có 'date' nhưng dtype là object -> đáng ngờ
    suspicious_dates = [
        c for c in df.columns
        if 'date' in c.lower() and df[c].dtype == 'object'
    ]
    if suspicious_dates:
        log(f'[CAUTION] Cột tên có "date" nhưng dtype object: {suspicious_dates}')
        for col in suspicious_dates:
            sample = df[col].dropna().head(3).tolist()
            log(f'   {col} sample: {sample}')


## 2. Schema và Dtypes


### `products.csv`
```
product_id        int64
product_name        str
category            str
segment             str
size                str
color               str
price           float64
cogs            float64
```

### `customers.csv`
```
customer_id            int64
zip                    int64
city                     str
signup_date              str
gender                   str
age_group                str
acquisition_channel      str
```

### `promotions.csv`
```
promo_id                   str
promo_name                 str
promo_type                 str
discount_value         float64
start_date                 str
end_date                   str
applicable_category        str
promo_channel              str
stackable_flag           int64
min_order_value          int64
```

### `geography.csv`
```
zip         int64
city          str
region        str
district      str
```

### `orders.csv`
```
order_id          int64
order_date          str
customer_id 

**Kết luận Cell 2:** Kiểm tra xem các cột `order_date`, `ship_date`, `delivery_date`, `return_date`, `review_date`, `signup_date`, `start_date`, `end_date`, `snapshot_date` có được đọc đúng kiểu `object` (string) hay không. Nếu có cột nào bị flag `[CAUTION]`, cần điều tra thêm ở Cell 7.

---
## Cell 3 — Missing values

Đếm số null và tỷ lệ null theo từng cột cho toàn bộ 13 bảng. Chỉ in bảng nào có ít nhất một cột missing.

In [5]:
log('\n## 3. Missing Values\n')

for name, df in tables.items():
    miss = df.isna().sum()
    miss_pct = (df.isna().mean() * 100).round(2)
    miss_df = pd.DataFrame({'n_missing': miss, 'pct_missing': miss_pct})
    miss_df = miss_df[miss_df['n_missing'] > 0].sort_values('pct_missing', ascending=False)

    if len(miss_df) == 0:
        log(f'\n### `{name}` — không có missing')
    else:
        log(f'\n### `{name}` — {len(miss_df)} cột có missing:')
        log(f'```\n{miss_df.to_string()}\n```')


## 3. Missing Values


### `products` — không có missing

### `customers` — không có missing

### `promotions` — 1 cột có missing:
```
                     n_missing  pct_missing
applicable_category         40         80.0
```

### `geography` — không có missing

### `orders` — không có missing

### `order_items` — 2 cột có missing:
```
            n_missing  pct_missing
promo_id_2     714463        99.97
promo_id       438353        61.34
```

### `payments` — không có missing

### `shipments` — không có missing

### `returns` — không có missing

### `reviews` — không có missing

### `sales` — không có missing

### `inventory` — không có missing

### `web_traffic` — không có missing


**Kết luận Cell 3:** Phân loại các cột có missing thành hai nhóm sau khi đọc kết quả:
- **Null có nghĩa (by design):** `promo_id`, `promo_id_2`, `age_group`, `gender`, `acquisition_channel`, `promo_channel`, `applicable_category`, `min_order_value` — schema đã khai báo nullable.
- **Null bất thường (cần điều tra):** bất kỳ cột nào không thuộc danh sách trên mà vẫn có missing.

---
## Cell 4 — Categorical columns — toàn bộ unique values

Với cột kiểu `object` có ít hơn hoặc bằng 50 unique values, in toàn bộ `value_counts()` kèm tần suất. Không cap top 20 — in hết để thấy có giá trị lạ nào không.

In [6]:
def is_categorical_like(series, max_unique=50):
    """Cột categorical-like: dtype object/string và có tối đa max_unique giá trị."""
    if series.dtype not in ['object', 'string', 'category']:
        return False
    return series.nunique(dropna=True) <= max_unique

log('\n## 4. Categorical Values (unique <= 50)\n')

for name, df in tables.items():
    cat_cols = [c for c in df.columns if is_categorical_like(df[c])]
    if not cat_cols:
        continue
    log(f'\n### `{name}`')
    for col in cat_cols:
        counts = df[col].value_counts(dropna=False)
        log(f'\n#### `{name}.{col}` — {df[col].nunique()} unique values:')
        log(f'```\n{counts.to_string()}\n```')


## 4. Categorical Values (unique <= 50)


### `products`

#### `products.category` — 4 unique values:
```
category
Streetwear    1320
Outdoor        743
Casual         201
GenZ           148
```

#### `products.segment` — 8 unique values:
```
segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
```

#### `products.size` — 4 unique values:
```
size
S     603
M     603
L     603
XL    603
```

#### `products.color` — 10 unique values:
```
color
black     242
orange    242
green     241
silver    241
pink      241
yellow    241
red       241
blue      241
white     241
purple    241
```

### `customers`

#### `customers.city` — 42 unique values:
```
city
Cam Pha                4398
Thai Nguyen            4347
Phu Ly                 4243
Hanoi                  4240
Ha Long                4236
Bac Ninh               4172
Hai Phong              4170
Nam Dinh               4169
Bac Gian

**Kết luận Cell 4:** Kiểm tra các cột quan trọng:
- `order_status`: có đúng các giá trị `pending`, `shipped`, `delivered`, `cancelled`, `returned` không? Có giá trị nào lạ không?
- `payment_method`: phân phối có hợp lý không?
- `size`: có đúng `S`, `M`, `L`, `XL` không — hay còn giá trị khác?
- `return_reason`, `traffic_source`, `segment`, `category`, `region`: in hết để thấy toàn bộ tập giá trị thực tế.

---
## Cell 5 — High-cardinality string columns

Cột `object` có hơn 50 unique values (như `product_name`, `city`, `review_title`...). In top 10 và **bottom 10** theo tần suất. Bottom 10 đặc biệt hữu ích để phát hiện typo — giá trị xuất hiện đúng 1 lần (singleton) thường là lỗi nhập liệu.

In [7]:
log('\n## 5. High-cardinality String Columns\n')

for name, df in tables.items():
    str_cols = df.select_dtypes(include=['object', 'string']).columns
    high_card = [c for c in str_cols if df[c].nunique() > 50]
    if not high_card:
        continue
    log(f'\n### `{name}`')
    for col in high_card:
        n_uniq = df[col].nunique()
        n_singletons = (df[col].value_counts() == 1).sum()
        log(f'\n#### `{name}.{col}` — {n_uniq:,} unique values, {n_singletons:,} singletons (xuất hiện đúng 1 lần)')
        counts = df[col].value_counts(dropna=False)
        log('Top 10:')
        log(f'```\n{counts.head(10).to_string()}\n```')
        log('Bottom 10 (có thể là typo):')
        log(f'```\n{counts.tail(10).to_string()}\n```')


## 5. High-cardinality String Columns


### `products`

#### `products.product_name` — 2,172 unique values, 1,986 singletons (xuất hiện đúng 1 lần)
Top 10:
```
product_name
VietMode RP-01    3
VietMode RP-02    3
VietMode RP-03    3
VietMode RP-04    3
VietMode RP-05    3
VietMode RP-06    3
VietMode RP-07    3
VietMode RP-08    3
VietMode RP-09    3
VietMode RP-10    3
```
Bottom 10 (có thể là typo):
```
product_name
VietMode MP-23    1
VietMode MP-24    1
VietMode MP-25    1
VietMode MP-26    1
VietMode MP-27    1
VietMode MP-28    1
VietMode MP-29    1
VietMode MP-30    1
VietMode MP-31    1
VietMode MP-32    1
```

### `customers`

#### `customers.signup_date` — 3,941 unique values, 53 singletons (xuất hiện đúng 1 lần)
Top 10:
```
signup_date
2022-06-02    78
2022-11-13    77
2022-11-15    76
2021-10-31    75
2022-07-18    75
2022-11-30    74
2022-11-22    73
2022-10-25    73
2022-06-28    73
2021-10-05    72
```
Bottom 10 (có thể là typo):
```
signup_date
2012-02-05    1
2012-02-

**Kết luận Cell 5:** Chú ý cột nào có `n_singletons` cao bất thường so với tổng số dòng. Singleton không nhất thiết là lỗi (ví dụ `product_name` singleton có thể là sản phẩm ít bán), nhưng singleton trong `city` hay `district` thường là typo cần xử lý.

---
## Cell 6 — Numeric columns

`describe()` tiêu chuẩn cộng thêm hai cột bổ sung: `n_zeros` (số dòng bằng 0) và `n_negative` (số dòng âm). Cảnh báo rõ nếu có giá trị âm kèm sample để dễ tra cứu.

In [8]:
log('\n## 6. Numeric Columns Summary\n')

for name, df in tables.items():
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        continue
    log(f'\n### `{name}`')
    desc = df[num_cols].describe().T
    desc['n_zeros']    = (df[num_cols] == 0).sum().values
    desc['n_negative'] = (df[num_cols] < 0).sum().values
    log(f'```\n{desc.round(2).to_string()}\n```')

    # Cảnh báo cụ thể nếu có giá trị âm
    for col in num_cols:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            sample = df.loc[df[col] < 0, col].head(3).tolist()
            log(f'[CAUTION] `{col}` có {n_neg:,} giá trị âm — sample: {sample}')


## 6. Numeric Columns Summary


### `products`
```
             count     mean      std   min     25%      50%      75%      max  n_zeros  n_negative
product_id  2412.0  1206.50   696.43  1.00  603.75  1206.50  1809.25   2412.0        0           0
price       2412.0  4928.22  4776.74  9.06   59.44  4399.60  7720.51  40950.0        0           0
cogs        2412.0  3868.35  3878.58  5.18   35.07  3184.93  5864.92  38902.5        0           0
```

### `customers`
```
                count      mean       std     min       25%      50%        75%       max  n_zeros  n_negative
customer_id  121930.0  78736.90  45492.20     1.0  39343.50  78784.5  118156.75  157563.0        0           0
zip          121930.0  50990.17  26871.91  1001.0  28689.25  49835.0   73488.00   99950.0        0           0
```

### `promotions`
```
                 count      mean       std   min   25%   50%       75%       max  n_zeros  n_negative
discount_value    50.0     18.50     11.24  10.0  12.0  16.5      

**Kết luận Cell 6:** Các cột cần chú ý đặc biệt:
- `price`, `cogs`: có âm hoặc bằng 0 không?
- `quantity`, `unit_price` trong `order_items`: âm là lỗi rõ ràng.
- `payment_value`: âm hoặc bằng 0 không hợp lý với đơn hàng thực.
- `refund_amount` trong `returns`: âm là vô lý.
- `fill_rate`, `sell_through_rate`, `bounce_rate`: là tỷ lệ, kỳ vọng nằm trong `[0, 1]` — nếu nằm ngoài khoảng này thì bất thường.
- `Revenue`, `COGS` trong `sales`: ngày nào có giá trị âm hoặc bằng 0 cần đánh dấu.

---
## Cell 7 — Date columns

Tự phát hiện cột date dựa vào tên, thử parse với `errors='coerce'`, rồi in range và số lượng parse thất bại. Đặc biệt kiểm tra ngày trong tương lai — thường là lỗi nhập liệu.

In [9]:
log('\n## 7. Date Columns\n')

for name, df in tables.items():
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    if not date_cols:
        continue
    log(f'\n### `{name}`')
    for col in date_cols:
        try:
            parsed = pd.to_datetime(df[col], errors='coerce')
            # Số dòng parse thất bại = tổng NaN sau parse - NaN gốc
            n_original_null = df[col].isna().sum()
            n_parse_fail    = parsed.isna().sum() - n_original_null

            log(f'\n#### `{name}.{col}`')
            log(f'- Range      : {parsed.min()} -> {parsed.max()}')
            log(f'- Distinct   : {parsed.nunique():,} ngày khác nhau')
            log(f'- Parse fail : {n_parse_fail:,} dòng không parse được thành ngày')

            n_future = (parsed > pd.Timestamp.now()).sum()
            if n_future > 0:
                log(f'- [CAUTION] {n_future:,} ngày nằm trong tương lai')
        except Exception as e:
            log(f'  Không thể parse {col}: {e}')


## 7. Date Columns


### `customers`

#### `customers.signup_date`
- Range      : 2012-01-17 00:00:00 -> 2022-12-31 00:00:00
- Distinct   : 3,941 ngày khác nhau
- Parse fail : 0 dòng không parse được thành ngày

### `promotions`

#### `promotions.start_date`
- Range      : 2013-01-31 00:00:00 -> 2022-11-18 00:00:00
- Distinct   : 50 ngày khác nhau
- Parse fail : 0 dòng không parse được thành ngày

#### `promotions.end_date`
- Range      : 2013-03-01 00:00:00 -> 2022-12-31 00:00:00
- Distinct   : 50 ngày khác nhau
- Parse fail : 0 dòng không parse được thành ngày

### `orders`

#### `orders.order_date`
- Range      : 2012-07-04 00:00:00 -> 2022-12-31 00:00:00
- Distinct   : 3,833 ngày khác nhau
- Parse fail : 0 dòng không parse được thành ngày

### `shipments`

#### `shipments.ship_date`
- Range      : 2012-07-04 00:00:00 -> 2022-12-29 00:00:00
- Distinct   : 3,831 ngày khác nhau
- Parse fail : 0 dòng không parse được thành ngày

#### `shipments.delivery_date`
- Range      : 2012-07-06

**Kết luận Cell 7:** Ghi nhận range thực tế của từng bảng để so sánh với khai báo trong đề (04/07/2012 - 31/12/2022 cho train, 01/01/2023 - 01/07/2024 cho test). Cột nào có `parse_fail > 0` cần điều tra thêm — đó là giá trị không phải ngày tháng hợp lệ cần xử lý trước khi dùng.

---
## Cell 8 — Primary key candidates

Tự phát hiện cột có thể là primary key dựa vào tên (kết thúc bằng `_id` hoặc là `zip`). Kiểm tra xem cột đó có unique hoàn toàn không.

In [10]:
log('\n## 8. Primary Key Candidates\n')
log('| Table | Column | Is unique? | n_distinct | n_rows | n_dups |')
log('|-------|--------|-----------|------------|--------|--------|')

for name, df in tables.items():
    id_cols = [c for c in df.columns if c.endswith('_id') or c == 'zip']
    for col in id_cols:
        n_dist  = df[col].nunique()
        n_rows  = len(df)
        n_dups  = n_rows - n_dist
        is_uniq = df[col].is_unique
        flag    = '[OK]' if is_uniq else '[DUP]'
        log(f'| {flag} {name} | {col} | {is_uniq} | {n_dist:,} | {n_rows:,} | {n_dups:,} |')


## 8. Primary Key Candidates

| Table | Column | Is unique? | n_distinct | n_rows | n_dups |
|-------|--------|-----------|------------|--------|--------|
| [OK] products | product_id | True | 2,412 | 2,412 | 0 |
| [OK] customers | customer_id | True | 121,930 | 121,930 | 0 |
| [DUP] customers | zip | False | 31,491 | 121,930 | 90,439 |
| [OK] promotions | promo_id | True | 50 | 50 | 0 |
| [OK] geography | zip | True | 39,948 | 39,948 | 0 |
| [OK] orders | order_id | True | 646,945 | 646,945 | 0 |
| [DUP] orders | customer_id | False | 90,246 | 646,945 | 556,699 |
| [DUP] orders | zip | False | 29,932 | 646,945 | 617,013 |
| [DUP] order_items | order_id | False | 646,945 | 714,669 | 67,724 |
| [DUP] order_items | product_id | False | 1,598 | 714,669 | 713,071 |
| [DUP] order_items | promo_id | False | 50 | 714,669 | 714,619 |
| [OK] payments | order_id | True | 646,945 | 646,945 | 0 |
| [OK] shipments | order_id | True | 566,067 | 566,067 | 0 |
| [OK] returns | return_id | True | 39,9

**Kết luận Cell 8:** Các cột kỳ vọng là primary key duy nhất:
- `orders.order_id`, `payments.order_id` (1:1 với orders), `customers.customer_id`, `products.product_id`, `returns.return_id`, `reviews.review_id`.
- Bảng `order_items` không có PK đơn — duplicate theo `(order_id, product_id)` là bình thường nếu cùng sản phẩm đặt nhiều lần trong một đơn.
- Bảng `inventory`: PK là composite `(snapshot_date, product_id)`.

Cột nào bị flag `[DUP]` mà kỳ vọng là PK cần đưa vào danh sách clean.

---
## Cell 9 — Foreign key orphans

Với mỗi cặp quan hệ FK -> PK, đếm số dòng trong bảng con có giá trị FK **không tồn tại** trong bảng cha. Orphan rows sẽ bị mất khi join — cần biết trước.

In [11]:
# Định nghĩa các cặp quan hệ FK dựa vào schema trong đề
FK_RELATIONS = [
    # (child_table, child_col, parent_table, parent_col)
    ('orders',     'customer_id', 'customers', 'customer_id'),
    ('orders',     'zip',         'geography', 'zip'),
    ('order_items','order_id',    'orders',    'order_id'),
    ('order_items','product_id',  'products',  'product_id'),
    ('payments',   'order_id',    'orders',    'order_id'),
    ('shipments',  'order_id',    'orders',    'order_id'),
    ('returns',    'order_id',    'orders',    'order_id'),
    ('returns',    'product_id',  'products',  'product_id'),
    ('reviews',    'order_id',    'orders',    'order_id'),
    ('reviews',    'product_id',  'products',  'product_id'),
    ('reviews',    'customer_id', 'customers', 'customer_id'),
    ('inventory',  'product_id',  'products',  'product_id'),
    ('customers',  'zip',         'geography', 'zip'),
]

log('\n## 9. Foreign Key Orphans\n')
log('| Child.col | -> Parent.col | n_orphan_rows | pct_orphans |')
log('|-----------|---------------|---------------|-------------|')

for child_t, child_c, parent_t, parent_c in FK_RELATIONS:
    if child_t not in tables or parent_t not in tables:
        continue
    child_vals  = set(tables[child_t][child_c].dropna())
    parent_vals = set(tables[parent_t][parent_c].dropna())
    orphan_vals = child_vals - parent_vals
    n_orphan_rows = tables[child_t][child_c].isin(orphan_vals).sum()
    pct = n_orphan_rows / len(tables[child_t]) * 100
    flag = '[ORPHAN]' if n_orphan_rows > 0 else '[OK]'
    log(f'| {flag} {child_t}.{child_c} | {parent_t}.{parent_c} | {n_orphan_rows:,} | {pct:.2f}% |')


## 9. Foreign Key Orphans

| Child.col | -> Parent.col | n_orphan_rows | pct_orphans |
|-----------|---------------|---------------|-------------|
| [OK] orders.customer_id | customers.customer_id | 0 | 0.00% |
| [OK] orders.zip | geography.zip | 0 | 0.00% |
| [OK] order_items.order_id | orders.order_id | 0 | 0.00% |
| [OK] order_items.product_id | products.product_id | 0 | 0.00% |
| [OK] payments.order_id | orders.order_id | 0 | 0.00% |
| [OK] shipments.order_id | orders.order_id | 0 | 0.00% |
| [OK] returns.order_id | orders.order_id | 0 | 0.00% |
| [OK] returns.product_id | products.product_id | 0 | 0.00% |
| [OK] reviews.order_id | orders.order_id | 0 | 0.00% |
| [OK] reviews.product_id | products.product_id | 0 | 0.00% |
| [OK] reviews.customer_id | customers.customer_id | 0 | 0.00% |
| [OK] inventory.product_id | products.product_id | 0 | 0.00% |
| [OK] customers.zip | geography.zip | 0 | 0.00% |


**Kết luận Cell 9:** Nếu không có dòng nào bị flag `[ORPHAN]` thì tính toàn vẹn tham chiếu của bộ dữ liệu là tốt. Nếu có, cần quyết định xử lý theo một trong hai hướng: drop các orphan rows, hoặc giữ lại và chấp nhận mất dữ liệu khi join.

---
## Cell 10 — Cardinality thực tế giữa các bảng

Đề bài khai báo các quan hệ (1:1, 1:N...). Cell này verify bằng data thật — không tin vào spec cho đến khi data xác nhận.

In [12]:
def check_cardinality(left_df, left_key, right_df, right_key):
    """Trả về cardinality thực tế giữa 2 bảng theo key."""
    left_per_key  = left_df.groupby(left_key).size()
    right_per_key = right_df.groupby(right_key).size()
    return {
        'left_max'  : int(left_per_key.max())  if len(left_per_key)  > 0 else 0,
        'right_max' : int(right_per_key.max()) if len(right_per_key) > 0 else 0,
    }

PAIRS = [
    ('orders',   'order_id', 'payments',   'order_id'),
    ('orders',   'order_id', 'shipments',  'order_id'),
    ('orders',   'order_id', 'order_items','order_id'),
    ('orders',   'order_id', 'returns',    'order_id'),
    ('orders',   'order_id', 'reviews',    'order_id'),
    ('products', 'product_id','inventory', 'product_id'),
]

log('\n## 10. Cardinality giữa các bảng (thực tế từ data)\n')
log('| Left table | Right table | left_max_per_key | right_max_per_key | Cardinality |')
log('|------------|-------------|-----------------|-------------------|-------------|')

for lt, lk, rt, rk in PAIRS:
    if lt not in tables or rt not in tables:
        continue
    info = check_cardinality(tables[lt], lk, tables[rt], rk)
    if   info['left_max'] == 1 and info['right_max'] == 1:
        card = '1:1'
    elif info['left_max'] == 1 and info['right_max'] > 1:
        card = '1:N'
    elif info['left_max'] > 1 and info['right_max'] == 1:
        card = 'N:1'
    else:
        card = 'N:M'
    log(f'| {lt}.{lk} | {rt}.{rk} | {info["left_max"]} | {info["right_max"]} | **{card}** |')


## 10. Cardinality giữa các bảng (thực tế từ data)

| Left table | Right table | left_max_per_key | right_max_per_key | Cardinality |
|------------|-------------|-----------------|-------------------|-------------|
| orders.order_id | payments.order_id | 1 | 1 | **1:1** |
| orders.order_id | shipments.order_id | 1 | 1 | **1:1** |
| orders.order_id | order_items.order_id | 1 | 5 | **1:N** |
| orders.order_id | returns.order_id | 1 | 3 | **1:N** |
| orders.order_id | reviews.order_id | 1 | 3 | **1:N** |
| products.product_id | inventory.product_id | 1 | 126 | **1:N** |


**Kết luận Cell 10:** So sánh với khai báo trong đề:
- `orders <-> payments` kỳ vọng `1:1` — nếu ra `1:N` hoặc `N:M` thì có vấn đề.
- `orders <-> shipments` kỳ vọng `1:0 hoặc 1` — nếu ra `1:N` thì một đơn có nhiều shipment, cần xem lại.
- `orders <-> order_items` kỳ vọng `1:N` — bình thường.
- `products <-> inventory` kỳ vọng `1:N` — bình thường (1 sản phẩm có nhiều snapshot theo tháng).

---
## Cell 11 — Full duplicate rows

Kiểm tra dòng nào giống nhau **100% tất cả các cột**. Đây là rác rõ ràng nhất — không có lý do nghiệp vụ nào để tồn tại.

In [13]:
log('\n## 11. Full Duplicate Rows\n')
log('| Table | n_full_duplicates | pct |')
log('|-------|-------------------|-----|')

for name, df in tables.items():
    n_dup = df.duplicated().sum()
    pct   = n_dup / len(df) * 100 if len(df) > 0 else 0
    flag  = '[DUP]' if n_dup > 0 else '[OK]'
    log(f'| {flag} {name} | {n_dup:,} | {pct:.4f}% |')


## 11. Full Duplicate Rows

| Table | n_full_duplicates | pct |
|-------|-------------------|-----|
| [OK] products | 0 | 0.0000% |
| [OK] customers | 0 | 0.0000% |
| [OK] promotions | 0 | 0.0000% |
| [OK] geography | 0 | 0.0000% |
| [OK] orders | 0 | 0.0000% |
| [OK] order_items | 0 | 0.0000% |
| [OK] payments | 0 | 0.0000% |
| [OK] shipments | 0 | 0.0000% |
| [OK] returns | 0 | 0.0000% |
| [OK] reviews | 0 | 0.0000% |
| [OK] sales | 0 | 0.0000% |
| [OK] inventory | 0 | 0.0000% |
| [OK] web_traffic | 0 | 0.0000% |


**Kết luận Cell 11:** Bảng nào có `n_full_duplicates > 0` sẽ được dedup trong bước clean (`.drop_duplicates()`). Chú ý tỷ lệ — nếu một bảng có >1% duplicate thì có thể có vấn đề ở pipeline tạo ra bộ dữ liệu, không chỉ là lỗi nhỏ.

---
## Cell 12 — Sanity checks theo logic thuần

Kiểm tra các bất biến logic chung mà bất kỳ bộ dữ liệu nào cũng nên thỏa — không dựa vào đặc thù của đề bài, chỉ dựa vào common sense.

In [14]:
log('\n## 12. Logical Sanity Checks\n')

# 12.1 products: cogs < price (ràng buộc trong đề, nhưng verify thực tế)
if 'products' in tables:
    p = tables['products']
    if {'cogs', 'price'}.issubset(p.columns):
        n_bad      = (p['cogs'] >= p['price']).sum()
        n_neg_price = (p['price'] <= 0).sum()
        n_neg_cogs  = (p['cogs'] < 0).sum()
        log(f'### products')
        log(f'- cogs >= price          : {n_bad:,} dòng')
        log(f'- price <= 0             : {n_neg_price:,} dòng')
        log(f'- cogs < 0               : {n_neg_cogs:,} dòng')

# 12.2 shipments: ship_date <= delivery_date
if 'shipments' in tables:
    s = tables['shipments'].copy()
    if {'ship_date', 'delivery_date'}.issubset(s.columns):
        s['ship_date']     = pd.to_datetime(s['ship_date'],     errors='coerce')
        s['delivery_date'] = pd.to_datetime(s['delivery_date'], errors='coerce')
        n_bad = (s['ship_date'] > s['delivery_date']).sum()
        log(f'\n### shipments')
        log(f'- ship_date > delivery_date : {n_bad:,} dòng')

# 12.3 promotions: start_date <= end_date
if 'promotions' in tables:
    pr = tables['promotions'].copy()
    if {'start_date', 'end_date'}.issubset(pr.columns):
        pr['start_date'] = pd.to_datetime(pr['start_date'], errors='coerce')
        pr['end_date']   = pd.to_datetime(pr['end_date'],   errors='coerce')
        n_bad = (pr['start_date'] > pr['end_date']).sum()
        log(f'\n### promotions')
        log(f'- start_date > end_date : {n_bad:,} dòng')

# 12.4 orders: order_date >= signup_date (khách phải đăng ký trước khi đặt hàng)
if {'orders', 'customers'}.issubset(tables):
    o = tables['orders'].copy()
    c = tables['customers'].copy()
    if {'order_date'}.issubset(o.columns) and {'signup_date'}.issubset(c.columns):
        o['order_date']  = pd.to_datetime(o['order_date'],  errors='coerce')
        c['signup_date'] = pd.to_datetime(c['signup_date'], errors='coerce')
        merged = o.merge(c[['customer_id', 'signup_date']], on='customer_id', how='left')
        n_bad  = (merged['order_date'] < merged['signup_date']).sum()
        log(f'\n### orders vs customers')
        log(f'- order_date < signup_date (đặt hàng trước khi đăng ký?) : {n_bad:,} dòng')

# 12.5 returns: refund_amount >= 0
if 'returns' in tables:
    r = tables['returns']
    if 'refund_amount' in r.columns:
        n_neg  = (r['refund_amount'] < 0).sum()
        n_zero = (r['refund_amount'] == 0).sum()
        log(f'\n### returns')
        log(f'- refund_amount < 0 : {n_neg:,} dòng')
        log(f'- refund_amount = 0 : {n_zero:,} dòng')

# 12.6 order_items: quantity > 0, unit_price > 0, discount_amount >= 0
if 'order_items' in tables:
    oi = tables['order_items']
    log(f'\n### order_items')
    if 'quantity' in oi.columns:
        log(f'- quantity <= 0       : {(oi["quantity"] <= 0).sum():,} dòng')
    if 'unit_price' in oi.columns:
        log(f'- unit_price <= 0     : {(oi["unit_price"] <= 0).sum():,} dòng')
    if 'discount_amount' in oi.columns:
        log(f'- discount_amount < 0 : {(oi["discount_amount"] < 0).sum():,} dòng')

# 12.7 inventory: fill_rate và sell_through_rate nằm trong [0, 1]
if 'inventory' in tables:
    inv = tables['inventory']
    log(f'\n### inventory')
    for rate_col in ['fill_rate', 'sell_through_rate']:
        if rate_col in inv.columns:
            n_out = ((inv[rate_col] < 0) | (inv[rate_col] > 1)).sum()
            log(f'- {rate_col} ngoài [0,1] : {n_out:,} dòng')

# 12.8 web_traffic: bounce_rate nằm trong [0, 1]
if 'web_traffic' in tables:
    wt = tables['web_traffic']
    if 'bounce_rate' in wt.columns:
        n_out = ((wt['bounce_rate'] < 0) | (wt['bounce_rate'] > 1)).sum()
        log(f'\n### web_traffic')
        log(f'- bounce_rate ngoài [0,1] : {n_out:,} dòng')


## 12. Logical Sanity Checks

### products
- cogs >= price          : 0 dòng
- price <= 0             : 0 dòng
- cogs < 0               : 0 dòng

### shipments
- ship_date > delivery_date : 0 dòng

### promotions
- start_date > end_date : 0 dòng

### orders vs customers
- order_date < signup_date (đặt hàng trước khi đăng ký?) : 477,453 dòng

### returns
- refund_amount < 0 : 0 dòng
- refund_amount = 0 : 0 dòng

### order_items
- quantity <= 0       : 0 dòng
- unit_price <= 0     : 0 dòng
- discount_amount < 0 : 0 dòng

### inventory
- fill_rate ngoài [0,1] : 0 dòng
- sell_through_rate ngoài [0,1] : 0 dòng

### web_traffic
- bounce_rate ngoài [0,1] : 0 dòng


**Kết luận Cell 12:** Bất kỳ check nào trả về kết quả `> 0` đều cần được đưa vào danh sách thảo luận của nhóm trước khi clean. Không tự sửa ở đây — chỉ ghi nhận.

---
## Cell 13 — Crosstab shipments / returns theo order_status

In crosstab để thấy phân phối thực tế — **không assume** trước status nào hợp lệ hay không. Để nhóm tự đọc và quyết định rule.

In [15]:
log('\n## 13. Crosstab order_status vs shipments / returns\n')

# 13.1 shipments vs order_status
if {'shipments', 'orders'}.issubset(tables):
    o = tables['orders'][['order_id', 'order_status']]
    s = tables['shipments'][['order_id']].copy()
    s['has_shipment'] = 1

    merged = o.merge(s.drop_duplicates('order_id'), on='order_id', how='left')
    merged['has_shipment'] = merged['has_shipment'].fillna(0).astype(int)

    ct = pd.crosstab(
        merged['order_status'],
        merged['has_shipment'],
        margins=True,
        margins_name='Total'
    )
    ct.columns = [str(c) for c in ct.columns]
    log('### order_status vs has_shipment')
    log('(0 = không có shipment, 1 = có shipment)')
    log(f'```\n{ct.to_string()}\n```')
    log('\nDe team doc: status nao nen co shipment? Status nao khong nen co?')

# 13.2 returns vs order_status
if {'returns', 'orders'}.issubset(tables):
    o = tables['orders'][['order_id', 'order_status']]
    r = tables['returns'][['order_id']].copy()
    r['has_return'] = 1

    merged = o.merge(r.drop_duplicates('order_id'), on='order_id', how='left')
    merged['has_return'] = merged['has_return'].fillna(0).astype(int)

    ct = pd.crosstab(
        merged['order_status'],
        merged['has_return'],
        margins=True,
        margins_name='Total'
    )
    ct.columns = [str(c) for c in ct.columns]
    log('\n### order_status vs has_return')
    log('(0 = không có return, 1 = có return)')
    log(f'```\n{ct.to_string()}\n```')
    log('\nDe team doc: status nao nen co return? Co bat thuong nao khong?')


## 13. Crosstab order_status vs shipments / returns

### order_status vs has_shipment
(0 = không có shipment, 1 = có shipment)
```
                  0       1   Total
order_status                       
cancelled     59462       0   59462
created        7275       0    7275
delivered       524  516192  516716
paid          13577       0   13577
returned         29   36113   36142
shipped          11   13762   13773
Total         80878  566067  646945
```

De team doc: status nao nen co shipment? Status nao khong nen co?

### order_status vs has_return
(0 = không có return, 1 = có return)
```
                   0      1   Total
order_status                       
cancelled      59462      0   59462
created         7275      0    7275
delivered     516716      0  516716
paid           13577      0   13577
returned          80  36062   36142
shipped        13773      0   13773
Total         610883  36062  646945
```

De team doc: status nao nen co return? Co bat thuong nao khong?


**Kết luận Cell 13:** Theo schema trong đề, `shipments` chỉ tồn tại cho đơn `shipped`, `delivered`, `returned`. Nếu crosstab cho thấy đơn `cancelled` hay `pending` cũng có shipment thì đó là bất thường cần xử lý. Tương tự, `returns` chỉ nên xuất hiện với đơn `returned`.

---
## Cell 14 — Gap detection trong time series sales.csv

Phần 3 của cuộc thi là bài toán forecasting doanh thu theo ngày. Chuỗi thời gian **không được có ngày bị thiếu** — nếu thiếu ngày thì mô hình sẽ học sai chu kỳ và xu hướng. Cell này kiểm tra `sales.csv` xem có ngày nào bị thiếu không trong khoảng train.

In [16]:
log('\n## 14. Gap Detection trong Time Series sales.csv\n')

if 'sales' in tables:
    sales = tables['sales'].copy()

    if 'Date' in sales.columns:
        sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
        sales = sales.dropna(subset=['Date'])
        sales = sales.sort_values('Date')

        date_min  = sales['Date'].min()
        date_max  = sales['Date'].max()
        n_actual  = sales['Date'].nunique()
        full_range = pd.date_range(start=date_min, end=date_max, freq='D')
        n_expected = len(full_range)
        n_missing  = n_expected - n_actual

        log(f'- Ngày đầu tiên  : {date_min.date()}')
        log(f'- Ngày cuối cùng : {date_max.date()}')
        log(f'- Số ngày thực tế: {n_actual:,}')
        log(f'- Số ngày kỳ vọng (liên tục): {n_expected:,}')
        log(f'- Số ngày bị thiếu: {n_missing:,}')

        if n_missing > 0:
            missing_dates = full_range[~full_range.isin(sales['Date'])]
            log(f'\n[CAUTION] {n_missing} ngày bị thiếu trong chuỗi thời gian:')
            log(f'```\n{pd.Series(missing_dates).dt.date.to_string()}\n```')
        else:
            log('\n- Chuỗi thời gian liên tục, không có ngày nào bị thiếu.')

        # Kiểm tra thêm: ngày nào có Revenue = 0 hoặc âm
        if 'Revenue' in sales.columns:
            n_zero_rev = (sales['Revenue'] == 0).sum()
            n_neg_rev  = (sales['Revenue'] < 0).sum()
            log(f'\n- Ngày có Revenue = 0 : {n_zero_rev:,}')
            log(f'- Ngày có Revenue < 0 : {n_neg_rev:,}')
            if n_zero_rev > 0:
                zero_dates = sales.loc[sales['Revenue'] == 0, 'Date'].dt.date.tolist()
                log(f'  Các ngày Revenue = 0: {zero_dates}')
else:
    log('Không tìm thấy bảng sales.')


## 14. Gap Detection trong Time Series sales.csv

- Ngày đầu tiên  : 2012-07-04
- Ngày cuối cùng : 2022-12-31
- Số ngày thực tế: 3,833
- Số ngày kỳ vọng (liên tục): 3,833
- Số ngày bị thiếu: 0

- Chuỗi thời gian liên tục, không có ngày nào bị thiếu.

- Ngày có Revenue = 0 : 0
- Ngày có Revenue < 0 : 0


**Kết luận Cell 14:** Đây là check quan trọng nhất cho Phần 3. Nếu có ngày bị thiếu, cần quyết định chiến lược impute (forward fill, interpolate, hay để mô hình tự xử lý bằng date features). Ngày có `Revenue = 0` cũng cần xem xét — có thể là ngày nghỉ lễ hợp lệ, hoặc là lỗi dữ liệu.

---
## Cell 15 — Lưu report ra file markdown

In [17]:
report_path = OUT_DIR / 'profiling_report.md'
report_path.write_text(REPORT_BUFFER.getvalue(), encoding='utf-8')

print(f'Report saved: {report_path}')
print(f'Size        : {report_path.stat().st_size / 1024:.1f} KB')
print()
print('Buoc tiep theo:')
print('  1. Mo file profiling_report.md, doc ky tung section.')
print('  2. Danh dau cac bat thuong [CAUTION] va [DUP] va [ORPHAN].')
print('  3. Hop nhom -- list ra cac rule clean cu the.')
print('  4. Moi viet clean_data.py dua tren cac rule do.')

Report saved: ..\data\interim\profiling_report.md
Size        : 40.0 KB

Buoc tiep theo:
  1. Mo file profiling_report.md, doc ky tung section.
  2. Danh dau cac bat thuong [CAUTION] va [DUP] va [ORPHAN].
  3. Hop nhom -- list ra cac rule clean cu the.
  4. Moi viet clean_data.py dua tren cac rule do.


---
## Tổng kết — Khám phá nhanh dữ liệu

Notebook này thực hiện đầy đủ 15 bước profiling trên toàn bộ 15 file CSV gốc. Kết quả được ghi ra `data/interim/profiling_report.md` để cả nhóm đọc chung.

**Checklist những điểm cần thảo luận sau khi chạy notebook:**

| Hạng mục | Cell | Cần làm gì |
|----------|------|------------|
| Cột date bị đọc thành object | 2 | Điều tra giá trị không hợp lệ, quyết định parse hay drop |
| Null bất thường (ngoài danh sách nullable) | 3 | Quyết định impute hay drop |
| Giá trị categorical lạ | 4 | Quyết định map hay drop |
| Singleton trong high-cardinality columns | 5 | Đánh giá có phải typo không |
| Giá trị âm trong cột số | 6 | Quyết định sửa hay drop |
| Parse fail trong cột date | 7 | Điều tra thêm |
| PK bị duplicate | 8 | Dedup theo rule nào |
| FK orphan rows | 9 | Drop hay giữ |
| Cardinality không khớp spec | 10 | Điều tra nguyên nhân |
| Full duplicate rows | 11 | Drop toàn bộ |
| Vi phạm sanity checks | 12 | Xử lý từng case |
| Status không hợp lệ có shipment/return | 13 | Quyết định rule lọc |
| Ngày thiếu trong sales.csv | 14 | Chọn chiến lược impute cho forecasting |

**Nguyên tắc:** Không thay đổi bất cứ gì trong thư mục `data/raw/`. Mọi quyết định clean đều được viết thành code có thể tái lập trong `notebooks`, output ra `data/processed/`.

In [18]:
# Lấy toàn bộ nội dung đã thu thập được trong buffer
full_output = REPORT_BUFFER.getvalue()

# Lưu toàn bộ output thành một file Markdown hoàn chỉnh
report_path = OUT_DIR / 'DataExploration_report.md'

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(full_output)

print(f"Đã xuất toàn bộ báo cáo ra file: {report_path}")

Đã xuất toàn bộ báo cáo ra file: ..\data\interim\DataExploration_report.md
